# Notebook 02 - Cleaning & Preparation

Cleaning and preparing raw transport data for analysis.

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
validations = pd.read_csv("../data/interim/validations_raw.csv")
routes = pd.read_csv("../data/interim/routes_raw.csv")
vehicles = pd.read_csv("../data/interim/vehicles_raw.csv")
stops = pd.read_csv("../data/interim/stops_raw.csv")

## Remove duplicates

We remove exact duplicate rows and duplicated validation_id values.
For duplicated validation_id we keep the first occurrence.

In [ ]:
# Check duplicates before cleaning

print("Exact duplicates:", validations.duplicated().sum())
print("Duplicate validation_id:", validations["validation_id"].duplicated().sum())

Exact duplicates: 802
Duplicate validation_id: 1600


In [5]:
# Save exact duplicates
duplicate_rows = validations[validations.duplicated()].copy()
duplicate_rows["reject_reason"] = "Exact duplicate"

# Remove exact duplicates
validations = validations.drop_duplicates()

# Save duplicate validation_id (except first)
duplicate_validation_id = validations[
    validations.duplicated(subset="validation_id", keep="first")
].copy()

duplicate_validation_id["reject_reason"] = "Duplicate validation_id"

# Remove duplicate validation_id
validations = validations.drop_duplicates(
    subset="validation_id",
    keep="first"
)

print("Rows after duplicate removal:", len(validations))

Rows after duplicate removal: 80000


## Convert validation timestamp

The validation_ts column contains mixed date formats.
It will be converted into a datetime format for time-based analysis.

In [6]:
# Save rows with unparseable validation_ts
invalid_ts = validations[
    pd.to_datetime(
        validations["validation_ts"],
        format="mixed",
        errors="coerce"
    ).isna()
].copy()

invalid_ts["reject_reason"] = "Unparseable validation_ts"

# Parse validation_ts
validations["validation_ts"] = pd.to_datetime(
    validations["validation_ts"],
    format="mixed",
    errors="coerce"
)

# Drop invalid datetime rows
validations = validations[
    validations["validation_ts"].notna()
].copy()

print("Unparseable validation_ts:", len(invalid_ts))
print("Remaining validations:", len(validations))
invalid_ts["reject_reason"] = "Unparseable validation_ts"

# Parse validation_ts
validations["validation_ts"] = pd.to_datetime(
    validations["validation_ts"],
    format="mixed",
    errors="coerce"
)

# Drop invalid datetime rows
validations = validations[
    validations["validation_ts"].notna()
].copy()

print("Unparseable validation_ts:", len(invalid_ts))
print("Remaining validations:", len(validations))

Unparseable validation_ts: 0
Remaining validations: 80000
Unparseable validation_ts: 0
Remaining validations: 80000


In [8]:
id_columns = [
    "validation_id",
    "route_id",
    "vehicle_id",
    "stop_id",
    "card_id"
]

for col in id_columns:
    validations[col] = validations[col].astype("string").str.strip()

print("Whitespace removed from ID columns.")

Whitespace removed from ID columns.


In [10]:
fare_type_mapping = {
    "Adult": "Adult",
    "Full Fare": "Adult",
    "Student": "Student",
    "Senior": "Senior",
    "Senior Citizen": "Senior",
    "Child": "Child",
    "Kid": "Child",
    "Pass Holder": "Pass Holder",
    "Passholder": "Pass Holder",
    "Monthly Pass": "Pass Holder"
}

validations["fare_type"] = (
    validations["fare_type"]
    .astype("string")
    .str.strip()
    .map(fare_type_mapping)
)

print(validations["fare_type"].value_counts(dropna=False))

fare_type
Adult          44401
Student        11847
Senior          9471
Pass Holder     7958
Child           6323
Name: count, dtype: int64


In [11]:
transfer_mapping = {
    "Y": True,
    "N": False,
    "Yes": True,
    "No": False,
    "1": True,
    "0": False,
    "True": True,
    "False": False
}

validations["transfer_flag"] = (
    validations["transfer_flag"]
    .astype("string")
    .str.strip()
    .map(transfer_mapping)
    .astype("boolean")
)

print(validations["transfer_flag"].value_counts(dropna=False))

transfer_flag
False    44085
True     28075
<NA>      7840
Name: count, dtype: Int64


## Clean ID columns

Remove leading and trailing whitespace from identifier columns.

In [ ]:
id_columns = [
    "validation_id",
    "route_id",
    "vehicle_id",
    "stop_id",
    "card_id"
]

for col in id_columns:
    validations[col] = validations[col].str.strip()

## Normalize fare_type

Standardize different fare type values into a fixed set of categories.

In [ ]:
validations["fare_type"] = (
    validations["fare_type"]
    .str.strip()
    .str.lower()
)

fare_mapping = {
    "adult": "Adult",
    "full fare": "Adult",
    "student": "Student",
    "senior": "Senior",
    "child": "Child",
    "pass holder": "Pass Holder"
}

validations["fare_type"] = validations["fare_type"].map(fare_mapping)

validations["fare_type"].value_counts(dropna=False)

fare_type
Adult          44401
Student        11847
NaN             9305
Senior          6304
Child           4157
Pass Holder     3986
Name: count, dtype: int64

In [ ]:
validations["fare_type"].isna().sum()

np.int64(9305)

In [ ]:
raw_validations = pd.read_csv("../data/interim/validations_raw.csv")

raw_validations["fare_type"].value_counts(dropna=False)

fare_type
Full Fare         11405
ADULT             11317
Adult             11282
adult             11280
student            4077
STUDENT            4062
Student            3961
Senior             3251
Senior Citizen     3237
senior             3179
Kid                2202
child              2124
Child              2109
Pass Holder        2075
PassHolder         2032
Monthly Pass       2012
pass holder        1995
Name: count, dtype: int64

In [ ]:
raw_validations = pd.read_csv("../data/interim/validations_raw.csv")

validations["fare_type"] = raw_validations["fare_type"]

In [ ]:
validations["fare_type"] = (
    validations["fare_type"]
    .str.strip()
    .str.lower()
)

fare_mapping = {
    "adult": "Adult",
    "full fare": "Adult",
    
    "student": "Student",
    
    "senior": "Senior",
    "senior citizen": "Senior",
    
    "child": "Child",
    "kid": "Child",
    
    "pass holder": "Pass Holder",
    "passholder": "Pass Holder",
    "monthly pass": "Pass Holder"
}

validations["fare_type"] = validations["fare_type"].map(fare_mapping)

validations["fare_type"].value_counts(dropna=False)

fare_type
Adult          44401
Student        11847
Senior          9471
Pass Holder     7958
Child           6323
Name: count, dtype: int64

## Normalize transfer_flag

Convert different transfer flag formats into a boolean column.

In [12]:
transfer_mapping = {
    "Y": True,
    "y": True,
    "1": True,
    "TRUE": True,
    "True": True,
    "true": True,
    
    "N": False,
    "n": False,
    "0": False,
    "FALSE": False,
    "False": False,
    "false": False
}

validations["transfer_flag"] = (
    validations["transfer_flag"]
    .astype("string")
    .str.strip()
    .map(transfer_mapping)
    .astype("boolean")
)

print(validations["transfer_flag"].value_counts(dropna=False))

transfer_flag
False    44085
True     28075
<NA>      7840
Name: count, dtype: Int64


In [13]:
raw_validations = pd.read_csv("../data/raw/validations.csv")

raw_validations["transfer_flag"].value_counts(dropna=False)

transfer_flag
N        36888
Y        24596
0         8075
FALSE     5593
1         4053
TRUE      2395
Name: count, dtype: int64

In [14]:
print(validations.shape)
print(raw_validations.shape)

(80000, 10)
(81600, 10)


In [15]:
transfer_mapping = {
    "N": False,
    "Y": True,
    "0": False,
    "1": True,
    "FALSE": False,
    "TRUE": True
}

transfer_values = (
    raw_validations[["validation_id", "transfer_flag"]]
    .copy()
)

transfer_values["validation_id"] = (
    transfer_values["validation_id"]
    .astype("string")
    .str.strip()
)

transfer_values["transfer_flag"] = (
    transfer_values["transfer_flag"]
    .astype("string")
    .str.strip()
    .map(transfer_mapping)
)

validations = validations.drop(columns="transfer_flag")

validations = validations.merge(
    transfer_values,
    on="validation_id",
    how="left"
)

validations["transfer_flag"] = validations["transfer_flag"].astype("boolean")

print(validations["transfer_flag"].value_counts(dropna=False))

transfer_flag
False    50556
True     31044
Name: count, dtype: Int64


In [16]:
missing_passenger = validations[
    validations["passenger_count"].isna()
].copy()

missing_passenger["reject_reason"] = "Missing passenger_count"

print("Missing passenger_count:", len(missing_passenger))

Missing passenger_count: 833


In [17]:
validations = validations[
    validations["passenger_count"].notna()
].copy()

print("Remaining validations:", len(validations))

Remaining validations: 80767


In [18]:
print("Current validations:", validations.shape)
print("Duplicate validation_id:", validations["validation_id"].duplicated().sum())
print("Exact duplicates:", validations.duplicated().sum())

Current validations: (80767, 10)
Duplicate validation_id: 1578
Exact duplicates: 1578


In [19]:
# Remove duplicates again after the transfer_flag merge
validations = validations.drop_duplicates(
    subset="validation_id",
    keep="first"
).copy()

print("After removing duplicate validation_id:", validations.shape)
print("Duplicate validation_id:", validations["validation_id"].duplicated().sum())

After removing duplicate validation_id: (79189, 10)
Duplicate validation_id: 0


In [20]:
validations["is_reversed"] = (
    validations["passenger_count"] < 0
)

print(
    "Negative passenger_count:",
    validations["is_reversed"].sum()
)

Negative passenger_count: 358


In [21]:
is_reversed = True

In [22]:
missing_fare = validations[
    validations["fare_amount"].isna()
].copy()

missing_fare["reject_reason"] = "Missing fare_amount"

print("Missing fare_amount:", len(missing_fare))

Missing fare_amount: 1156


In [23]:
validations["fare_amount"] = pd.to_numeric(
    validations["fare_amount"],
    errors="coerce"
)

print(validations["fare_amount"].dtype)

float64


In [24]:
fare_medians = validations.groupby("fare_type")["fare_amount"].transform("median")

validations["fare_amount"] = validations["fare_amount"].fillna(fare_medians)

print("Missing fare_amount after imputation:",
      validations["fare_amount"].isna().sum())

Missing fare_amount after imputation: 0


In [25]:
fare_cap = validations["fare_amount"].quantile(0.995)

print("99.5th percentile:", fare_cap)

99.5th percentile: 2.86


Why 99.5th percentile capping?
The 99.5th percentile was used to limit the influence of unusually high fare amounts while preserving the vast majority of valid observations. Values above this threshold are capped rather than removed, so no additional validation records are lost.

Why cap fare_amount at the 99.5th percentile?
Values above the 99.5th percentile are treated as extreme outliers. Instead of removing these valid observations, capping them preserves the rows while limiting the influence of unusually large fares on aggregate analysis. The 99.5th percentile was selected because it removes the influence of only the most extreme 0.5% of values while retaining 99.5% of the observed fare distribution.

### Why cap fare_amount at the 99.5th percentile?

Values above the 99.5th percentile are treated as extreme outliers.
Instead of removing these observations, capping preserves the rows while
reducing the influence of unusually large fares on the analysis.
The 99.5th percentile keeps 99.5% of the observed fare distribution
and limits the effect of only the most extreme 0.5% of values.


In [27]:
validations["card_id"] = (
    validations["card_id"]
    .astype("string")
    .str.strip()
)

validations["card_id"] = validations["card_id"].replace(
    ["", "nan", "None", "<NA>"],
    pd.NA
)

validations["card_id"] = validations["card_id"].fillna("SINGLE_RIDE")

print(
    "SINGLE_RIDE count:",
    (validations["card_id"] == "SINGLE_RIDE").sum()
)

SINGLE_RIDE count: 9448


In [28]:
routes["route_name"] = (
    routes["route_name"]
    .astype("string")
    .str.strip()
    .str.title()
)

routes["mode"] = (
    routes["mode"]
    .astype("string")
    .str.strip()
    .str.title()
)

routes["district"] = (
    routes["district"]
    .astype("string")
    .str.strip()
    .fillna("Unknown")
)

print(routes[["route_name", "mode", "district"]].head())

             route_name   mode     district
0     Angel Harbor Line   Tram  Harbourside
1   Jeffrey Street Line    Bus      Unknown
2     Erica Plains Line  Metro    Southbank
3  Brittany Bypass Line   Tram    Northgate
4     Clayton Fort Line    Bus      Unknown


In [29]:
route_length_medians = routes.groupby("mode")["route_length_km"].transform("median")

routes["route_length_km"] = routes["route_length_km"].fillna(
    route_length_medians
)

print(
    "Missing route_length_km after imputation:",
    routes["route_length_km"].isna().sum()
)

Missing route_length_km after imputation: 0


In [31]:
vehicles.loc[
    ~vehicles["year_manufactured"].between(1980, 2026),
    "year_manufactured"
] = pd.NA

print(
    "Invalid years replaced with null:",
    vehicles["year_manufactured"].isna().sum()
)

Invalid years replaced with null: 15


In [32]:
current_year = 2026

vehicles["vehicle_age"] = (
    current_year - vehicles["year_manufactured"]
)

vehicles["vehicle_age_band"] = pd.cut(
    vehicles["vehicle_age"],
    bins=[-1, 5, 10, 20, float("inf")],
    labels=["0-5", "6-10", "11-20", "20+"]
)

vehicles["vehicle_age_band"] = (
    vehicles["vehicle_age_band"]
    .astype("string")
    .fillna("Unknown")
)

print(vehicles["vehicle_age_band"].value_counts(dropna=False))

vehicle_age_band
20+        84
11-20      44
6-10       23
Unknown    15
0-5        14
Name: count, dtype: Int64


In [33]:
capacity_medians = vehicles.groupby("vehicle_type")["capacity"].transform("median")

vehicles["capacity"] = vehicles["capacity"].fillna(
    capacity_medians
)

print(
    "Missing capacity after imputation:",
    vehicles["capacity"].isna().sum()
)

Missing capacity after imputation: 0


In [34]:
stops["stop_name"] = (
    stops["stop_name"]
    .astype("string")
    .str.strip()
    .str.title()
)

print(stops["stop_name"].head())

0    Matthew Radial Stop
1    Brianna Avenue Stop
2       Jean Circle Stop
3     Carlos Bypass Stop
4     Jones Freeway Stop
Name: stop_name, dtype: string


In [35]:
shelter_mapping = {
    "Y": True,
    "y": True,
    "1": True,
    "TRUE": True,
    "True": True,
    "true": True,
    
    "N": False,
    "n": False,
    "0": False,
    "FALSE": False,
    "False": False,
    "false": False
}

stops["has_shelter"] = (
    stops["has_shelter"]
    .astype("string")
    .str.strip()
    .map(shelter_mapping)
    .astype("boolean")
)

print(stops["has_shelter"].value_counts(dropna=False))

has_shelter
False    211
True     189
Name: count, dtype: Int64


In [36]:
invalid_reference = (
    ~validations["route_id"].isin(routes["route_id"])
    | ~validations["vehicle_id"].isin(vehicles["vehicle_id"])
    | ~validations["stop_id"].isin(stops["stop_id"])
)

print("Invalid references:", invalid_reference.sum())

Invalid references: 40


In [37]:
rejected_validations = validations[invalid_reference].copy()

rejected_validations["reject_reason"] = "Invalid route_id, vehicle_id, or stop_id"

print("Rejected validations:", rejected_validations.shape)

Rejected validations: (40, 12)


In [38]:
validations = validations[~invalid_reference].copy()

print("Remaining validations:", validations.shape)

Remaining validations: (79149, 11)


In [39]:
validations["date"] = validations["validation_ts"].dt.date
validations["year"] = validations["validation_ts"].dt.year
validations["month"] = validations["validation_ts"].dt.month

validations["year_month"] = (
    validations["validation_ts"].dt.to_period("M").astype(str)
)

validations["quarter"] = (
    validations["validation_ts"].dt.quarter
)

print(validations[
    ["validation_ts", "date", "year", "month", "year_month", "quarter"]
].head())

        validation_ts        date  year  month year_month  quarter
0 2024-05-13 18:54:29  2024-05-13  2024      5    2024-05        2
1 2024-06-06 08:48:00  2024-06-06  2024      6    2024-06        2
2 2024-06-03 13:15:00  2024-06-03  2024      6    2024-06        2
3 2024-04-21 17:30:00  2024-04-21  2024      4    2024-04        2
4 2024-06-14 18:35:27  2024-06-14  2024      6    2024-06        2


In [40]:
validations["hour"] = validations["validation_ts"].dt.hour

validations["day_of_week"] = validations["validation_ts"].dt.day_name()

validations["is_weekend"] = (
    validations["validation_ts"].dt.dayofweek >= 5
)

validations["time_period"] = pd.cut(
    validations["hour"],
    bins=[-1, 3, 6, 9, 15, 18, 23],
    labels=[
        "Night",
        "Early",
        "AM Peak",
        "Midday",
        "PM Peak",
        "Evening"
    ]
)

print(validations[
    ["validation_ts", "hour", "day_of_week",
     "is_weekend", "time_period"]
].head())

        validation_ts  hour day_of_week  is_weekend time_period
0 2024-05-13 18:54:29    18      Monday       False     PM Peak
1 2024-06-06 08:48:00     8    Thursday       False     AM Peak
2 2024-06-03 13:15:00    13      Monday       False      Midday
3 2024-04-21 17:30:00    17      Sunday        True     PM Peak
4 2024-06-14 18:35:27    18      Friday       False     PM Peak


In [41]:
validations["revenue"] = (
    validations["fare_amount"] * validations["passenger_count"]
)

print(validations[
    ["fare_amount", "passenger_count", "revenue"]
].head())

   fare_amount  passenger_count  revenue
0         2.53              1.0     2.53
1         2.48              1.0     2.48
2         2.85              1.0     2.85
3         1.64              3.0     4.92
4         1.34              1.0     1.34


In [42]:
all_rejected = pd.concat(
    [
        missing_passenger,
        rejected_validations
    ],
    ignore_index=True
)

print("Total rejected rows:", all_rejected.shape)
print(all_rejected["reject_reason"].value_counts())

Total rejected rows: (873, 12)
reject_reason
Missing passenger_count                     833
Invalid route_id, vehicle_id, or stop_id     40
Name: count, dtype: int64


In [43]:
from pathlib import Path

interim_dir = Path("../data/interim")
rejects_dir = Path("../data/rejects")

interim_dir.mkdir(parents=True, exist_ok=True)
rejects_dir.mkdir(parents=True, exist_ok=True)

all_rejected.to_csv(
    rejects_dir / "rejected_validations.csv",
    index=False
)

print("Rejected rows saved:", len(all_rejected))
print("Saved to:", rejects_dir / "rejected_validations.csv")

Rejected rows saved: 873
Saved to: ..\data\rejects\rejected_validations.csv


In [44]:
validations.to_csv(
    interim_dir / "validations_clean.csv",
    index=False
)

routes.to_csv(
    interim_dir / "routes_clean.csv",
    index=False
)

vehicles.to_csv(
    interim_dir / "vehicles_clean.csv",
    index=False
)

stops.to_csv(
    interim_dir / "stops_clean.csv",
    index=False
)

print("Cleaned files saved successfully.")
print("Validations:", validations.shape)
print("Routes:", routes.shape)
print("Vehicles:", vehicles.shape)
print("Stops:", stops.shape)

Cleaned files saved successfully.
Validations: (79149, 21)
Routes: (60, 6)
Vehicles: (180, 7)
Stops: (400, 5)


In [48]:
def normalize_fare_type(df):
    df = df.copy()

    mapping = {
        "Full Fare": "Adult",
        "Senior Citizen": "Senior",
        "Kid": "Child",
        "Passholder": "Pass Holder",
        "Monthly Pass": "Pass Holder"
    }

    df["fare_type"] = df["fare_type"].replace(mapping)

    return df

In [49]:
validations = normalize_fare_type(validations)

In [50]:
drop_counts = {}

In [51]:
drop_counts["missing passenger_count"] = 833
drop_counts["invalid references"] = 40

In [52]:
print("Rows dropped by reason:")
for reason, count in drop_counts.items():
    print(f"{reason}: {count}")

print("Total rows dropped:", sum(drop_counts.values()))

Rows dropped by reason:
missing passenger_count: 833
invalid references: 40
Total rows dropped: 873


In [53]:
drop_counts = {
    "exact duplicates": 802,
    "duplicate validation_id": 1600,
    "missing passenger_count": 833,
    "invalid references": 40
}

print("Rows dropped by reason:")

for reason, count in drop_counts.items():
    print(f"{reason}: {count}")

print("Total rows dropped:", sum(drop_counts.values()))

Rows dropped by reason:
exact duplicates: 802
duplicate validation_id: 1600
missing passenger_count: 833
invalid references: 40
Total rows dropped: 3275


In [55]:
def normalize_fare_type(df):
    df = df.copy()

    fare_mapping = {
        "Full Fare": "Adult",
        "Senior Citizen": "Senior",
        "Kid": "Child",
        "Passholder": "Pass Holder",
        "Monthly Pass": "Pass Holder"
    }

    df["fare_type"] = df["fare_type"].replace(fare_mapping)

    return df

In [56]:
validations = normalize_fare_type(validations)

print(validations["fare_type"].value_counts())

fare_type
Adult          43917
Student        11726
Senior          9369
Pass Holder     7880
Child           6257
Name: count, dtype: int64


In [57]:
def normalize_transfer_flag(df):
    df = df.copy()

    transfer_mapping = {
        "Y": True,
        "y": True,
        "1": True,
        "TRUE": True,
        "True": True,
        "true": True,
        "N": False,
        "n": False,
        "0": False,
        "FALSE": False,
        "False": False,
        "false": False
    }

    df["transfer_flag"] = (
        df["transfer_flag"]
        .astype("string")
        .str.strip()
        .map(transfer_mapping)
        .astype("boolean")
    )

    return df

In [58]:
validations = normalize_transfer_flag(validations)

print(validations["transfer_flag"].value_counts(dropna=False))

transfer_flag
False    49030
True     30119
Name: count, dtype: Int64


In [59]:
def handle_passenger_count(df):
    df = df.copy()

    df["is_reversed"] = df["passenger_count"] < 0

    return df

In [60]:
validations = handle_passenger_count(validations)

print("Negative passenger_count:", validations["is_reversed"].sum())
print("Missing passenger_count:", validations["passenger_count"].isna().sum())

Negative passenger_count: 358
Missing passenger_count: 0


In [61]:
def handle_fare_amount(df):
    df = df.copy()

    # Impute missing fare_amount with median for each fare_type
    df["fare_amount"] = (
        df.groupby("fare_type")["fare_amount"]
        .transform(lambda x: x.fillna(x.median()))
    )

    # Cap extreme values at the 99.5th percentile
    percentile_995 = df["fare_amount"].quantile(0.995)

    df["fare_amount"] = df["fare_amount"].clip(upper=percentile_995)

    return df

In [62]:
validations = handle_fare_amount(validations)

print("Missing fare_amount:", validations["fare_amount"].isna().sum())
print("99.5th percentile:", validations["fare_amount"].quantile(0.995))

Missing fare_amount: 0
99.5th percentile: 2.86


In [63]:
def normalize_card_id(df):
    df = df.copy()

    df["card_id"] = (
        df["card_id"]
        .astype("string")
        .str.strip()
        .replace("", "SINGLE_RIDE")
        .fillna("SINGLE_RIDE")
    )

    return df

In [64]:
validations = normalize_card_id(validations)

print("SINGLE_RIDE count:", 
      (validations["card_id"] == "SINGLE_RIDE").sum())

SINGLE_RIDE count: 9441


In [65]:
def clean_routes(df):
    df = df.copy()

    # Trim and title-case route_name
    df["route_name"] = (
        df["route_name"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    # Standardise mode
    df["mode"] = (
        df["mode"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    # Fill missing district
    df["district"] = df["district"].fillna("Unknown")

    return df

In [66]:
routes = clean_routes(routes)

print(routes[["route_name", "mode", "district"]].head())

             route_name   mode     district
0     Angel Harbor Line   Tram  Harbourside
1   Jeffrey Street Line    Bus      Unknown
2     Erica Plains Line  Metro    Southbank
3  Brittany Bypass Line   Tram    Northgate
4     Clayton Fort Line    Bus      Unknown


In [ ]:
def fill_route_length(df):
    df = df.copy()

    df["route_length_km"] = (
        df.groupby("mode")["route_length_km"]
        .transform(lambda x: x.fillna(x.median()))
    )

    return df

In [69]:
def fill_route_length(df):
    df = df.copy()

    df["route_length_km"] = (
        df.groupby("mode")["route_length_km"]
        .transform(lambda x: x.fillna(x.median()))
    )

    return df

In [70]:
routes = fill_route_length(routes)

print(
    "Missing route_length_km after imputation:",
    routes["route_length_km"].isna().sum()
)

Missing route_length_km after imputation: 0


In [71]:
def clean_vehicle_year(df):
    df = df.copy()

    # Invalid years become null
    df.loc[
        ~df["year_manufactured"].between(1980, 2026),
        "year_manufactured"
    ] = pd.NA

    # Derive vehicle age band
    current_year = 2026
    age = current_year - df["year_manufactured"]

    df["vehicle_age_band"] = pd.cut(
        age,
        bins=[-1, 5, 10, 20, float("inf")],
        labels=["0-5", "6-10", "11-20", "20+"]
    ).astype("string")

    df["vehicle_age_band"] = df["vehicle_age_band"].fillna("Unknown")

    return df

In [72]:
vehicles = clean_vehicle_year(vehicles)

print(
    "Invalid years replaced with null:",
    vehicles["year_manufactured"].isna().sum()
)

print(vehicles["vehicle_age_band"].value_counts(dropna=False))

Invalid years replaced with null: 15
vehicle_age_band
20+        84
11-20      44
6-10       23
Unknown    15
0-5        14
Name: count, dtype: Int64


In [73]:
def fill_vehicle_capacity(df):
    df = df.copy()

    df["capacity"] = (
        df.groupby("vehicle_type")["capacity"]
        .transform(lambda x: x.fillna(x.median()))
    )

    return df

In [74]:
vehicles = fill_vehicle_capacity(vehicles)

print(
    "Missing capacity after imputation:",
    vehicles["capacity"].isna().sum()
)

Missing capacity after imputation: 0


In [76]:
def clean_stops(df):
    df = df.copy()

    df["stop_name"] = (
        df["stop_name"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    shelter_mapping = {
        "Y": True,
        "y": True,
        "1": True,
        "TRUE": True,
        "True": True,
        "true": True,
        "N": False,
        "n": False,
        "0": False,
        "FALSE": False,
        "False": False,
        "false": False
    }

    df["has_shelter"] = (
        df["has_shelter"]
        .astype("string")
        .str.strip()
        .map(shelter_mapping)
        .astype("boolean")
    )

    return df

In [78]:
stops = clean_stops(stops)

print(stops["stop_name"].head())
print(stops["has_shelter"].value_counts(dropna=False))

0    Matthew Radial Stop
1    Brianna Avenue Stop
2       Jean Circle Stop
3     Carlos Bypass Stop
4     Jones Freeway Stop
Name: stop_name, dtype: string
has_shelter
False    211
True     189
Name: count, dtype: Int64


In [79]:
def check_referential_integrity(validations, routes, vehicles, stops):
    validations = validations.copy()

    valid_route_ids = set(routes["route_id"].dropna())
    valid_vehicle_ids = set(vehicles["vehicle_id"].dropna())
    valid_stop_ids = set(stops["stop_id"].dropna())

    invalid_mask = (
        ~validations["route_id"].isin(valid_route_ids)
        | ~validations["vehicle_id"].isin(valid_vehicle_ids)
        | ~validations["stop_id"].isin(valid_stop_ids)
    )

    rejected = validations.loc[invalid_mask].copy()
    rejected["reject_reason"] = "Invalid route_id, vehicle_id, or stop_id"

    cleaned = validations.loc[~invalid_mask].copy()

    return cleaned, rejected

In [80]:
validations, rejected_validations = check_referential_integrity(
    validations,
    routes,
    vehicles,
    stops
)

print("Rejected validations:", rejected_validations.shape)
print("Remaining validations:", validations.shape)

Rejected validations: (0, 22)
Remaining validations: (79149, 21)


In [81]:
invalid_refs = pd.read_csv(
    "../data/rejects/rejected_validations.csv"
)

invalid_refs = invalid_refs[
    invalid_refs["reject_reason"] == "Invalid route_id, vehicle_id, or stop_id"
]

print("Invalid reference rows loaded:", len(invalid_refs))

Invalid reference rows loaded: 40


In [82]:
validations_for_check = pd.concat(
    [validations, invalid_refs.drop(columns=["reject_reason"])],
    ignore_index=True
)

validations, rejected_validations = check_referential_integrity(
    validations_for_check,
    routes,
    vehicles,
    stops
)

print("Rejected validations:", rejected_validations.shape)
print("Remaining validations:", validations.shape)

Rejected validations: (40, 22)
Remaining validations: (79149, 21)


In [83]:
def add_validation_derived_columns(df):
    df = df.copy()

    df["date"] = df["validation_ts"].dt.date
    df["year"] = df["validation_ts"].dt.year
    df["month"] = df["validation_ts"].dt.month
    df["year_month"] = df["validation_ts"].dt.to_period("M").astype(str)
    df["quarter"] = df["validation_ts"].dt.quarter

    df["hour"] = df["validation_ts"].dt.hour
    df["day_of_week"] = df["validation_ts"].dt.day_name()
    df["is_weekend"] = df["validation_ts"].dt.dayofweek >= 5

    df["time_period"] = pd.cut(
        df["hour"],
        bins=[-1, 3, 6, 9, 15, 18, 23],
        labels=[
            "Night",
            "Early",
            "AM Peak",
            "Midday",
            "PM Peak",
            "Evening"
        ]
    )

    df["revenue"] = df["fare_amount"] * df["passenger_count"]

    return df

In [86]:
def add_validation_derived_columns(df):
    df = df.copy()

    # Make sure validation_ts is datetime
    df["validation_ts"] = pd.to_datetime(
        df["validation_ts"],
        format="mixed",
        errors="coerce"
    )

    df["date"] = df["validation_ts"].dt.date
    df["year"] = df["validation_ts"].dt.year
    df["month"] = df["validation_ts"].dt.month
    df["year_month"] = df["validation_ts"].dt.to_period("M").astype(str)
    df["quarter"] = df["validation_ts"].dt.quarter

    df["hour"] = df["validation_ts"].dt.hour
    df["day_of_week"] = df["validation_ts"].dt.day_name()
    df["is_weekend"] = df["validation_ts"].dt.dayofweek >= 5

    df["time_period"] = pd.cut(
        df["hour"],
        bins=[-1, 3, 6, 9, 15, 18, 23],
        labels=[
            "Night",
            "Early",
            "AM Peak",
            "Midday",
            "PM Peak",
            "Evening"
        ]
    )

    df["revenue"] = df["fare_amount"] * df["passenger_count"]

    return df

In [87]:
validations = add_validation_derived_columns(validations)

print(
    validations[
        [
            "validation_ts",
            "date",
            "year",
            "month",
            "year_month",
            "quarter",
            "hour",
            "day_of_week",
            "is_weekend",
            "time_period",
            "revenue"
        ]
    ].head()
)

        validation_ts        date  year  month year_month  quarter  hour  \
0 2024-05-13 18:54:29  2024-05-13  2024      5    2024-05        2    18   
1 2024-06-06 08:48:00  2024-06-06  2024      6    2024-06        2     8   
2 2024-06-03 13:15:00  2024-06-03  2024      6    2024-06        2    13   
3 2024-04-21 17:30:00  2024-04-21  2024      4    2024-04        2    17   
4 2024-06-14 18:35:27  2024-06-14  2024      6    2024-06        2    18   

  day_of_week  is_weekend time_period  revenue  
0      Monday       False     PM Peak     2.53  
1    Thursday       False     AM Peak     2.48  
2      Monday       False      Midday     2.85  
3      Sunday        True     PM Peak     4.92  
4      Friday       False     PM Peak     1.34  


In [88]:
from pathlib import Path

interim_dir = Path("../data/interim")
rejects_dir = Path("../data/rejects")

interim_dir.mkdir(parents=True, exist_ok=True)
rejects_dir.mkdir(parents=True, exist_ok=True)

validations.to_csv(
    interim_dir / "validations_clean.csv",
    index=False
)

routes.to_csv(
    interim_dir / "routes_clean.csv",
    index=False
)

vehicles.to_csv(
    interim_dir / "vehicles_clean.csv",
    index=False
)

stops.to_csv(
    interim_dir / "stops_clean.csv",
    index=False
)

print("Final cleaned files saved.")
print("Validations:", validations.shape)
print("Routes:", routes.shape)
print("Vehicles:", vehicles.shape)
print("Stops:", stops.shape)

Final cleaned files saved.
Validations: (79149, 21)
Routes: (60, 6)
Vehicles: (180, 7)
Stops: (400, 5)


In [89]:
drop_counts = {}

def log_drop(reason, count):
    drop_counts[reason] = drop_counts.get(reason, 0) + count

In [90]:
# Record rows that were already dropped during cleaning
log_drop("missing passenger_count", 833)
log_drop("invalid references", 40)

print("Rows dropped by reason:")

for reason, count in drop_counts.items():
    print(f"{reason}: {count}")

print("Total logged dropped rows:", sum(drop_counts.values()))

Rows dropped by reason:
missing passenger_count: 833
invalid references: 40
Total logged dropped rows: 873


In [46]:
def normalize_fare_type(df):
    df = df.copy()

    fare_mapping = {
        "Full Fare": "Adult",
        "Senior Citizen": "Senior",
        "Kid": "Child",
        "Passholder": "Pass Holder",
        "Monthly Pass": "Pass Holder"
    }

    df["fare_type"] = (
        df["fare_type"]
        .replace(fare_mapping)
    )

    return df

## Handle passenger_count

Check missing and negative passenger counts.
Missing values will be removed and negative values will be marked as reversed validations.

In [ ]:
print("Missing passenger_count:")
print(validations["passenger_count"].isna().sum())

print("\nNegative passenger_count:")
print((validations["passenger_count"] < 0).sum())

Missing passenger_count:
811

Negative passenger_count:
358


In [ ]:
validations["is_reversed"] = validations["passenger_count"] < 0

In [ ]:
validations["is_reversed"].value_counts()

is_reversed
False    79642
True       358
Name: count, dtype: int64

In [ ]:
validations = validations.dropna(subset=["passenger_count"])

print(validations.shape)

(79189, 11)


## Clean fare_amount

Missing fare amounts are filled using the median fare for each fare type.
Extreme values above the 99.5 percentile are capped.

In [ ]:
# Check missing and extreme fare values

print("Missing fare_amount:")
print(validations["fare_amount"].isna().sum())

print("\nMaximum fare_amount:")
print(validations["fare_amount"].max())

Missing fare_amount:
1156

Maximum fare_amount:
9999.0


In [ ]:
# Fill missing fare_amount with median by fare_type

validations["fare_amount"] = (
    validations.groupby("fare_type")["fare_amount"]
    .transform(lambda x: x.fillna(x.median()))
)

In [ ]:
validations["fare_amount"].isna().sum()

np.int64(0)

In [ ]:
# Calculate 99.5 percentile

fare_limit = validations["fare_amount"].quantile(0.995)

print(fare_limit)

2.86


In [ ]:
# Cap extreme values

validations["fare_amount"] = validations["fare_amount"].clip(
    upper=fare_limit
)

print(validations["fare_amount"].max())

2.86


In [ ]:
validations["fare_amount"] = validations["fare_amount"].clip(
    upper=fare_limit
)

print(validations["fare_amount"].max())

2.86


## Handle missing card_id

Blank card IDs represent single-ride tickets.
They are replaced with SINGLE_RIDE.

In [ ]:
validations["card_id"] = validations["card_id"].fillna("SINGLE_RIDE")

validations["card_id"].isna().sum()

np.int64(0)

In [ ]:
validations["card_id"].value_counts().head()

card_id
SINGLE_RIDE    9448
CARD-003617      11
CARD-018695      11
CARD-006504      10
CARD-006720      10
Name: count, dtype: int64

## Clean routes data

Standardize route names, modes, districts and route length values.

In [ ]:
routes["mode"].value_counts(dropna=False)

mode
bus       11
tram       9
BUS        9
Tram       9
Bus        8
BUS        6
metro      3
TRAM       2
Metro      2
METRO      1
Name: count, dtype: int64

In [ ]:
routes["district"].isna().sum()

np.int64(3)

In [ ]:
routes["district"] = routes["district"].fillna("Unknown")

routes["district"].isna().sum()

np.int64(0)

In [ ]:
routes["mode"].value_counts(dropna=False)

mode
bus       11
tram       9
BUS        9
Tram       9
Bus        8
BUS        6
metro      3
TRAM       2
Metro      2
METRO      1
Name: count, dtype: int64

In [ ]:
routes["mode"] = (
    routes["mode"]
    .str.strip()
    .str.title()
)

routes["mode"].value_counts()

mode
Bus      34
Tram     20
Metro     6
Name: count, dtype: int64

## Clean route_name

Remove extra spaces and standardize route names using title case.

In [ ]:
routes["route_name"] = (
    routes["route_name"]
    .str.strip()
    .str.title()
)

routes["route_name"].head()

0       Angel Harbor Line
1     Jeffrey Street Line
2       Erica Plains Line
3    Brittany Bypass Line
4       Clayton Fort Line
Name: route_name, dtype: str

## Handle missing route_length_km

Missing route lengths will be filled using the median value for each transport mode.

In [1]:
import pandas as pd

validations = pd.read_csv("../data/interim/validations_raw.csv")
routes = pd.read_csv("../data/interim/routes_raw.csv")
vehicles = pd.read_csv("../data/interim/vehicles_raw.csv")
stops = pd.read_csv("../data/interim/stops_raw.csv")

In [5]:
print(routes.shape)

(60, 6)


In [6]:
routes["route_length_km"] = (
    routes.groupby("mode")["route_length_km"]
    .transform(lambda x: x.fillna(x.median()))
)

routes["route_length_km"].isna().sum()


np.int64(0)

## Clean vehicles data

Clean manufacturing years, create vehicle age bands, and fill missing capacity values.

In [7]:
print("Missing capacity:")
print(vehicles["capacity"].isna().sum())

print("\nInvalid manufacturing years:")
print(
    vehicles[
        (vehicles["year_manufactured"] < 1980) |
        (vehicles["year_manufactured"] > 2026)
    ]
)

Missing capacity:
12

Invalid manufacturing years:
    vehicle_id     vehicle_type  capacity  year_manufactured            depot
54       V0055     Standard Bus      69.0               1899      North Depot
56       V0057        Metro Set     596.0               1899      North Depot
65       V0066     Standard Bus      65.0               2035       East Depot
76       V0077  Articulated Bus     123.0               2035       East Depot
82       V0083     Standard Bus      71.0               2035  Riverside Depot
89       V0090     Standard Bus      78.0               1899    Central Depot
91       V0092     Standard Bus      67.0               2035    Central Depot
119      V0120     Standard Bus      63.0               2035    Central Depot
132      V0133     Standard Bus      77.0               2035      North Depot
142      V0143     Standard Bus      72.0               2035       East Depot
146      V0147  Articulated Bus     121.0               2035    Central Depot
149      V015

## Clean manufacturing year

Manufacturing years outside the valid range (1980–2026) are replaced with missing values.

In [8]:
vehicles["capacity"] = (
    vehicles.groupby("vehicle_type")["capacity"]
    .transform(lambda x: x.fillna(x.median()))
)

vehicles["capacity"].isna().sum()

np.int64(0)

In [3]:

import pandas as pd

validations = pd.read_csv("../data/interim/validations_raw.csv")
routes = pd.read_csv("../data/interim/routes_raw.csv")
vehicles = pd.read_csv("../data/interim/vehicles_raw.csv")
stops = pd.read_csv("../data/interim/stops_raw.csv")

In [8]:
vehicles.loc[
    (vehicles["year_manufactured"] < 1980) |
    (vehicles["year_manufactured"] > 2026),
    "year_manufactured"
] = pd.NA


In [9]:
vehicles["year_manufactured"].isna().sum()

np.int64(15)

## Fill missing capacity

Missing capacity values are filled with the median capacity for each vehicle type.

In [10]:
vehicles["capacity"] = (
    vehicles.groupby("vehicle_type")["capacity"]
    .transform(lambda x: x.fillna(x.median()))
)

print(vehicles["capacity"].isna().sum())

0


## Create vehicle age band

Vehicles are grouped into age categories based on their manufacturing year.
Vehicles with missing manufacturing year are labeled as Unknown.

In [11]:
current_year = 2026

vehicle_age = current_year - vehicles["year_manufactured"]

vehicles["vehicle_age_band"] = "Unknown"

vehicles.loc[vehicle_age <= 5, "vehicle_age_band"] = "0-5"
vehicles.loc[(vehicle_age >= 6) & (vehicle_age <= 10), "vehicle_age_band"] = "6-10"
vehicles.loc[(vehicle_age >= 11) & (vehicle_age <= 20), "vehicle_age_band"] = "11-20"
vehicles.loc[vehicle_age > 20, "vehicle_age_band"] = "20+"

vehicles["vehicle_age_band"].value_counts()

vehicle_age_band
20+        84
11-20      44
6-10       23
Unknown    15
0-5        14
Name: count, dtype: int64

## Clean stops data

Standardize stop names and convert shelter values to boolean.

In [12]:
stops["stop_name"] = (
    stops["stop_name"]
    .str.strip()
    .str.title()
)

stops["stop_name"].head()

0    Matthew Radial Stop
1    Brianna Avenue Stop
2       Jean Circle Stop
3     Carlos Bypass Stop
4     Jones Freeway Stop
Name: stop_name, dtype: str

In [14]:
stops["has_shelter"].value_counts(dropna=False)

has_shelter
0    115
Y    102
N     96
1     87
Name: count, dtype: int64

In [1]:
import pandas as pd

validations = pd.read_csv("../data/interim/validations_raw.csv")
routes = pd.read_csv("../data/interim/routes_raw.csv")
vehicles = pd.read_csv("../data/interim/vehicles_raw.csv")
stops = pd.read_csv("../data/interim/stops_raw.csv")

## Normalize has_shelter

Convert shelter values to boolean values (True/False).

In [3]:
shelter_mapping = {
    "Y": True,
    "1": True,
    "N": False,
    "0": False
}

stops["has_shelter"] = (
    stops["has_shelter"]
    .astype(str)
    .str.strip()
    .map(shelter_mapping)
)

stops["has_shelter"].value_counts(dropna=False)

has_shelter
False    211
True     189
Name: count, dtype: int64

## Referential integrity

Check that all route, vehicle and stop IDs in the validations table exist in the corresponding master tables.

In [4]:
invalid_routes = validations[
    ~validations["route_id"].isin(routes["route_id"])
]

print("Invalid route_id:", len(invalid_routes))

Invalid route_id: 12106


In [5]:
print(validations.shape)
print(routes.shape)

(81600, 10)
(60, 6)


In [6]:
validations["route_id"].head(10)


0    R001 
1     R058
2     R043
3    R026 
4     R003
5     R013
6     R031
7    R060 
8     R019
9    R018 
Name: route_id, dtype: str

In [7]:
invalid_routes.head(10)

,validation_id,validation_ts,route_id,vehicle_id,stop_id,card_id,fare_type,fare_amount,passenger_count,transfer_flag
0,VAL-0068918,2024-05-13 18:54:29,R001,V0044,S0234,CARD-001739,Full Fare,2.53,1.0,N
3,VAL-0004736,2024-04-21 17:30:00,R026,V0108,S0330,CARD-013092,student,1.64,3.0,Y
7,VAL-0073779,2024-04-14 17:01:25,R060,V0149,S0106,CARD-004767,Child,1.11,1.0,TRUE
9,VAL-0032523,03/06/2024 08:08,R018,V0058,S0313,CARD-002453,ADULT,2.58,1.0,FALSE
26,VAL-0008772,2024-03-12 17:31:26,R010,V0074,S0041,CARD-015893,ADULT,2.45,1.0,Y
35,VAL-0051220,2024-02-13 17:36:20,R060,V0039,S0153,CARD-024808,Adult,2.36,1.0,N
41,VAL-0017708,2024-04-12 10:50:54,R013,V0029,S0347,CARD-002197,STUDENT,1.23,1.0,N
43,VAL-0079671,"March 18, 2024 1:35 PM",R040,V0176,S0345,CARD-001575,senior,1.21,1.0,N
68,VAL-0032141,22/05/2024 09:57,R022,V0003,S0086,NaN,Full Fare,2.59,1.0,0
77,VAL-0062461,2024-05-15 07:27:48,R052,V0065,S0387,NaN,adult,2.59,1.0,Y


In [8]:
routes["route_id"].head(10)

0    R001
1    R002
2    R003
3    R004
4    R005
5    R006
6    R007
7    R008
8    R009
9    R010
Name: route_id, dtype: str

In [9]:
validations["route_id"].head(10)

0    R001 
1     R058
2     R043
3    R026 
4     R003
5     R013
6     R031
7    R060 
8     R019
9    R018 
Name: route_id, dtype: str

In [11]:
routes["route_id"] = routes["route_id"].str.strip()
validations["route_id"] = validations["route_id"].str.strip()

In [12]:
validations["route_id"] = validations["route_id"].str.strip()
routes["route_id"] = routes["route_id"].str.strip()

validations["vehicle_id"] = validations["vehicle_id"].str.strip()
vehicles["vehicle_id"] = vehicles["vehicle_id"].str.strip()

validations["stop_id"] = validations["stop_id"].str.strip()
stops["stop_id"] = stops["stop_id"].str.strip()

In [13]:
invalid_routes = validations[
    ~validations["route_id"].isin(routes["route_id"])
]

print("Invalid route_id:", len(invalid_routes))

Invalid route_id: 14


In [14]:

invalid_routes["route_id"].value_counts()

route_id
R999    14
Name: count, dtype: int64

## Remove invalid route IDs

Rows with route IDs that do not exist in the routes table are removed.

In [15]:
validations = validations[
    validations["route_id"].isin(routes["route_id"])
]

print(validations.shape)

(81586, 10)


In [16]:
invalid_vehicles = validations[
    ~validations["vehicle_id"].isin(vehicles["vehicle_id"])
]

print("Invalid vehicle_id:", len(invalid_vehicles))

Invalid vehicle_id: 13


In [17]:
invalid_vehicles["vehicle_id"].value_counts()

vehicle_id
V9999    13
Name: count, dtype: int64

In [24]:
rejected_vehicles = invalid_vehicles.copy()

validations = validations[
    validations["vehicle_id"].isin(vehicles["vehicle_id"])
]

print(validations.shape)

(81559, 10)


## Remove invalid vehicle IDs

Rows with vehicle IDs that do not exist in the vehicles table are removed.

In [19]:
invalid_stops = validations[
    ~validations["stop_id"].isin(stops["stop_id"])
]

print("Invalid stop_id:", len(invalid_stops))

Invalid stop_id: 14


In [20]:
rejected_stops = invalid_stops.copy()

validations = validations[
    validations["stop_id"].isin(stops["stop_id"])
]

print(validations.shape)

(81572, 10)


In [21]:
invalid_stops["stop_id"].value_counts()

stop_id
S9999    14
Name: count, dtype: int64

## Remove invalid stop IDs

Rows with stop IDs that do not exist in the stops table are removed.

In [22]:
rejected_stops = invalid_stops.copy()

validations = validations[
    validations["stop_id"].isin(stops["stop_id"])
]

print(validations.shape)

(81572, 10)


In [38]:
print("Invalid route_id:",
      (~validations["route_id"].isin(routes["route_id"])).sum())

print("Invalid vehicle_id:",
      (~validations["vehicle_id"].isin(vehicles["vehicle_id"])).sum())

print("Invalid stop_id:",
      (~validations["stop_id"].isin(stops["stop_id"])).sum())

Invalid route_id: 0
Invalid vehicle_id: 0
Invalid stop_id: 0


In [28]:
validations["card_id"].value_counts(dropna=False)

card_id
NaN            9769
CARD-003617      11
CARD-018695      11
CARD-008423      11
CARD-003307      11
               ... 
CARD-022042       1
CARD-014912       1
CARD-008832       1
CARD-005786       1
CARD-022922       1
Name: count, Length: 23502, dtype: int64

In [29]:
invalid_card = validations["card_id"].isna()

rejected_validations = validations[invalid_card]

validations = validations[~invalid_card]

In [30]:
print("Rejected:", rejected_validations.shape)
print("Validations after cleaning:", validations.shape)

Rejected: (9769, 10)
Validations after cleaning: (71790, 10)


In [31]:
validations["card_id"].value_counts(dropna=False)

card_id
CARD-003617    11
CARD-018695    11
CARD-008423    11
CARD-003307    11
CARD-012313    10
               ..
CARD-022042     1
CARD-014912     1
CARD-008832     1
CARD-005786     1
CARD-022922     1
Name: count, Length: 23501, dtype: int64

## Handle SINGLE_RIDE cards

Missing card IDs were replaced with `SINGLE_RIDE`.
These validations are considered valid and are retained during referential integrity checks.

In [32]:
validations[validations["card_id"] == "SINGLE_RIDE"].shape

(0, 10)

In [36]:
invalid_reference = (
    ~validations["route_id"].isin(routes["route_id"]) |
    ~validations["vehicle_id"].isin(vehicles["vehicle_id"]) |
    ~validations["stop_id"].isin(stops["stop_id"])
)

rejected_validations = validations[invalid_reference]

validations = validations[~invalid_reference]

In [37]:
print("Rejected validations:", rejected_validations.shape)
print("Remaining validations:", validations.shape)

Rejected validations: (0, 10)
Remaining validations: (71790, 10)


In [2]:
print(validations.shape)

(81600, 10)


In [3]:
print(validations["card_id"].value_counts().head())

card_id
CARD-003617    11
CARD-018695    11
CARD-008423    11
CARD-003307    11
CARD-012313    10
Name: count, dtype: int64


In [4]:
validations["card_id"].isna().sum()

np.int64(9777)

In [6]:
validations["card_id"] = validations["card_id"].fillna("SINGLE_RIDE")

In [7]:
validations["card_id"].isna().sum()

np.int64(0)

In [8]:
validations["card_id"].value_counts().head()

card_id
SINGLE_RIDE    9777
CARD-003617      11
CARD-018695      11
CARD-008423      11
CARD-003307      11
Name: count, dtype: int64

## Derived columns

Create additional date and time features from the validation timestamp and calculate revenue.

In [9]:
# Convert validation_ts to datetime
validations["validation_ts"] = pd.to_datetime(
    validations["validation_ts"],
    format="mixed"
)

# Date columns
validations["date"] = validations["validation_ts"].dt.date
validations["year"] = validations["validation_ts"].dt.year
validations["month"] = validations["validation_ts"].dt.month
validations["year_month"] = validations["validation_ts"].dt.strftime("%Y-%m")
validations["quarter"] = validations["validation_ts"].dt.quarter

# Hour
validations["hour"] = validations["validation_ts"].dt.hour

# Day of week
validations["day_of_week"] = validations["validation_ts"].dt.day_name()

# Weekend flag
validations["is_weekend"] = validations["validation_ts"].dt.dayofweek >= 5

# Time period
def time_period(hour):
    if 4 <= hour <= 6:
        return "Early"
    elif 7 <= hour <= 9:
        return "AM Peak"
    elif 10 <= hour <= 15:
        return "Midday"
    elif 16 <= hour <= 18:
        return "PM Peak"
    elif 19 <= hour <= 23:
        return "Evening"
    else:
        return "Night"

validations["time_period"] = validations["hour"].apply(time_period)

# Revenue
validations["revenue"] = (
    validations["fare_amount"] * validations["passenger_count"]
)

# Preview
validations.head()

,validation_id,validation_ts,route_id,vehicle_id,stop_id,card_id,fare_type,fare_amount,passenger_count,transfer_flag,date,year,month,year_month,quarter,hour,day_of_week,is_weekend,time_period,revenue
0,VAL-0068918,2024-05-13 18:54:29,R001,V0044,S0234,CARD-001739,Full Fare,2.53,1.0,N,2024-05-13,2024,5,2024-05,2,18,Monday,False,PM Peak,2.53
1,VAL-0044034,2024-06-06 08:48:00,R058,V0086,S0131,CARD-000729,ADULT,2.48,1.0,Y,2024-06-06,2024,6,2024-06,2,8,Thursday,False,AM Peak,2.48
2,VAL-0017450,2024-06-03 13:15:00,R043,V0177,S0251,CARD-020205,Adult,2.85,1.0,FALSE,2024-06-03,2024,6,2024-06,2,13,Monday,False,Midday,2.85
3,VAL-0004736,2024-04-21 17:30:00,R026,V0108,S0330,CARD-013092,student,1.64,3.0,Y,2024-04-21,2024,4,2024-04,2,17,Sunday,True,PM Peak,4.92
4,VAL-0053735,2024-06-14 18:35:27,R003,V0057,S0389,CARD-003251,Student,1.34,1.0,0,2024-06-14,2024,6,2024-06,2,18,Friday,False,PM Peak,1.34


## Save cleaned datasets

Save cleaned datasets to the `data/interim` folder and rejected records to the `data/rejects` folder.

In [9]:
print(type(validations))

<class 'pandas.DataFrame'>


In [4]:
validations.to_csv("../data/interim/validations_clean.csv", index=False)
routes.to_csv("../data/interim/routes_clean.csv", index=False)
vehicles.to_csv("../data/interim/vehicles_clean.csv", index=False)
stops.to_csv("../data/interim/stops_clean.csv", index=False)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.
